# Моделирование пользовательского GMV

## Идея

В ноутбуке строится модель прогнозирования суммарного GMV пользователя
на следующие 30 дней.

Для этого:

1. формируются признаки активности пользователя на нескольких временных окнах;
2. добавляются агрегаты за всю историю и признаки давности последней активности;
3. для нескольких временных точек создаются обучающие признаки и фактический будущий GMV;
4. качество сравнивается с наивными baseline-прогнозами;
5. обучается CatBoostRegressor на логарифме целевой переменной;
6. модель переобучается на всех доступных данных;
7. формируется файл с предсказаниями для 250 000 пользователей.

## Ожидаемый результат

В результате должны быть:

- временные фолды для проверки модели без использования будущей информации;
- таблица признаков для обучения и финального прогноза;
- значение RMSLE для baseline и CatBoost;
- обученная финальная модель;
- файл `submission.csv` со столбцами `user_id` и `predict`.

In [ ]:
!pip install catboost -q

import os
import shutil
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import polars as pl

from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/e-cup")
DATA_PATH = BASE_DIR / "data/train.parquet"

data = pl.read_parquet(DATA_PATH)

print(f"Rows: {data.height:,}")
print(f"Columns: {data.width}")
print(f"Users: {data['user_id'].n_unique():,}")
print(f"Period: {data['event_date'].min()} to {data['event_date'].max()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00
Mounted at /content/drive
Rows: 30,631,006
Columns: 18
Users: 250,000
Period: 2025-01-01 to 2026-02-13


## Настройки расчёта

Горизонт прогноза равен 30 дням, поскольку именно такой период используется
в условиях соревнования. Признаки рассчитываются за окна от 7 до 90 дней:
короткие окна отражают текущую активность, а длинные - более устойчивое поведение.

Данные обрабатываются батчами по пользователям, чтобы не загружать все промежуточные
таблицы признаков в память одновременно.

In [ ]:
HORIZON = 30
N_FOLDS = 4
BATCH_SIZE = 50_000

WINDOWS = [
    ("7d", 6, 0),
    ("14d", 13, 0),
    ("30d", 29, 0),
    ("60d", 59, 0),
    ("90d", 89, 0),
]

SUM_COLS = [
    "search",
    "cat",
    "has_search_to_cart",
    "has_search_to_ord",
    "has_cat_to_cart",
    "has_cat_to_ord",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "gmv_search",
    "gmv_cat",
    "to_cart",
    "to_ord",
    "gmv",
    "searches",
]

MAX_COLS = [
    "gmv",
    "to_ord",
    "to_cart",
    "searches",
]

FEATURES_DIR = BASE_DIR / "data/v3/features"

if FEATURES_DIR.exists():
    shutil.rmtree(FEATURES_DIR)

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

user_ids = data["user_id"].unique().sort().to_list()

n_users = len(user_ids)
n_batches = (n_users + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Users: {n_users:,}")
print(f"Batches: {n_batches}")
print(f"Feature windows: {[x[0] for x in WINDOWS]}")

Users: 250,000
Batches: 5
Feature windows: ['7d', '14d', '30d', '60d', '90d']


## Метрика и временные точки валидации

RMSLE рассчитывается в логарифмической шкале, как и метрика соревнования.
Anchor-даты разделены 14 днями, чтобы получить несколько независимых временных срезов
и проверить, насколько стабильно модель работает в разные периоды.

Последние 30 дней истории нельзя использовать для валидации, потому что для них
ещё не будет полного 30-дневного target.

In [ ]:
def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true), 0, None)
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return np.sqrt(
        mean_squared_error(
            np.log1p(y_true),
            np.log1p(y_pred)
        )
    )


def generate_cv_anchor_dates(
    data,
    prediction_horizon_days=30,
    stride_days=14,
    min_history_days=90,
    n_folds=4,
):
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()

    latest_anchor = max_date - timedelta(days=prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days=min_history_days - 1)

    n_steps = (latest_anchor - earliest_anchor).days // stride_days

    all_anchors = [
        latest_anchor - timedelta(days=i * stride_days)
        for i in range(n_steps + 1)
    ]

    all_anchors = sorted(all_anchors)

    return all_anchors[-n_folds:]


anchors = generate_cv_anchor_dates(
    data,
    prediction_horizon_days=HORIZON,
    stride_days=14,
    min_history_days=90,
    n_folds=N_FOLDS,
)

anchor_end = data["event_date"].max()

print(f"CV anchors: {anchors}")
print(f"Final anchor: {anchor_end}")
print(
    f"Final prediction period: "
    f"{anchor_end + timedelta(days=1)} to "
    f"{anchor_end + timedelta(days=HORIZON)}"
)

CV anchors: [datetime.date(2025, 12, 3), datetime.date(2025, 12, 17), datetime.date(2025, 12, 31), datetime.date(2026, 1, 14)]
Final anchor: 2026-02-13
Final prediction period: 2026-02-14 to 2026-03-15


## Агрегаты по временным окнам

Для каждого пользователя и каждого окна считаем:

- сумму всех основных показателей;
- количество активных дней;
- максимальный дневной GMV;
- максимальное количество заказов;
- максимальную активность поиска;
- максимальную активность корзины.

Особенно полезно, что мы используем не только gmv, но и все связанные с покупательским поведением показатели: Search, Catalog, Cart, Orders.

In [ ]:
def window_agg_exprs(anchor, windows):
    exprs = []

    for window_name, start_offset, end_offset in windows:
        window_start = anchor - timedelta(days=start_offset)
        window_end = anchor - timedelta(days=end_offset)

        mask = pl.col("event_date").is_between(
            window_start,
            window_end
        )

        for col in SUM_COLS:
            exprs.append(
                pl.when(mask)
                .then(pl.col(col))
                .otherwise(0)
                .sum()
                .alias(f"{col}_{window_name}")
            )

        exprs.append(
            pl.when(mask)
            .then(1)
            .otherwise(0)
            .sum()
            .alias(f"active_days_{window_name}")
        )

        for col in MAX_COLS:
            exprs.append(
                pl.when(mask)
                .then(pl.col(col))
                .otherwise(None)
                .max()
                .fill_null(0)
                .alias(f"max_{col}_{window_name}")
            )

    return exprs

## Признаки за всю историю

Помимо недавней активности, сохраняем накопленную статистику пользователя:
общий GMV, заказы, добавления в корзину, поисковые запросы и количество активных дней.

Также определяем даты последней покупки, корзины, поиска и любой активности.
Это позволяет рассчитать давность событий - важный показатель для прогноза будущего GMV.

In [ ]:
def history_agg_exprs():
    exprs = [
        pl.len().alias("lifetime_active_days"),
        pl.col("event_date").min().alias("first_activity_date"),
        pl.col("event_date").max().alias("last_activity_date"),
        pl.when(pl.col("to_ord") > 0)
        .then(pl.col("event_date"))
        .otherwise(None)
        .max()
        .alias("last_order_date"),
        pl.when(pl.col("to_cart") > 0)
        .then(pl.col("event_date"))
        .otherwise(None)
        .max()
        .alias("last_cart_date"),
        pl.when(pl.col("searches") > 0)
        .then(pl.col("event_date"))
        .otherwise(None)
        .max()
        .alias("last_search_date"),
    ]

    for col in [
        "gmv",
        "gmv_search",
        "gmv_cat",
        "to_ord",
        "to_cart",
        "searches",
        "search_to_ord",
        "cat_to_ord",
        "search_to_cart",
        "cat_to_cart",
    ]:
        exprs.append(
            pl.col(col).sum().alias(f"lifetime_{col}")
        )

    return exprs

## Генерация признаков пользователя

Теперь соединяем предыдущие две части.

Здесь же создаём производные признаки:

- средний чек;
- GMV на активный день;
- заказы на активный день;
- конверсии Search → Order и Cart → Order;
- долю Search/Catalog в GMV;
- отношение последних 7 дней к обычному темпу за - 30 дней;
- отношение последних 30 дней к предыдущим 30 дням;
- логарифмические признаки.


In [ ]:
def generate_features(data, anchor_dates, user_ids):
    max_back = max(window[1] for window in WINDOWS)

    data_batch = data.filter(
        pl.col("user_id").is_in(user_ids)
    )

    parts = []

    for anchor in anchor_dates:
        history = data_batch.filter(
            pl.col("event_date") <= anchor
        )

        if history.height == 0:
            continue

        known_users = history.select("user_id").unique()

        recent = history.filter(
            pl.col("event_date") >= anchor - timedelta(days=max_back)
        )

        recent_features = (
            recent
            .group_by("user_id")
            .agg(window_agg_exprs(anchor, WINDOWS))
        )

        history_features = (
            history
            .group_by("user_id")
            .agg(history_agg_exprs())
        )

        features = (
            known_users
            .join(recent_features, on="user_id", how="left")
            .join(history_features, on="user_id", how="left")
        )

        features = features.with_columns(
            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_activity_date")
            ).dt.total_days().cast(pl.Float64).alias("days_since_last_activity"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_order_date")
            ).dt.total_days().cast(pl.Float64).alias("days_since_last_order"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_cart_date")
            ).dt.total_days().cast(pl.Float64).alias("days_since_last_cart"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_search_date")
            ).dt.total_days().cast(pl.Float64).alias("days_since_last_search"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("first_activity_date")
            ).dt.total_days().cast(pl.Float64).alias("tenure_days"),

            pl.lit(anchor.month).cast(pl.Int8).alias("anchor_month"),
            pl.lit(anchor.weekday()).cast(pl.Int8).alias("anchor_weekday"),
        )

        features = features.with_columns(
            (
                pl.col("gmv_30d") /
                (pl.col("to_ord_30d") + 1e-6)
            ).alias("avg_order_value_30d"),

            (
                pl.col("gmv_90d") /
                (pl.col("to_ord_90d") + 1e-6)
            ).alias("avg_order_value_90d"),

            (
                pl.col("gmv_30d") /
                (pl.col("active_days_30d") + 1e-6)
            ).alias("gmv_per_active_day_30d"),

            (
                pl.col("to_ord_30d") /
                (pl.col("active_days_30d") + 1e-6)
            ).alias("orders_per_active_day_30d"),

            (
                pl.col("to_ord_90d") /
                (pl.col("active_days_90d") + 1e-6)
            ).alias("orders_per_active_day_90d"),

            (
                pl.col("gmv_90d") /
                (pl.col("active_days_90d") + 1e-6)
            ).alias("gmv_per_active_day_90d"),

            (
                pl.col("to_ord_30d") /
                (pl.col("searches_30d") + 1e-6)
            ).alias("order_search_rate_30d"),

            (
                pl.col("to_ord_30d") /
                (pl.col("to_cart_30d") + 1e-6)
            ).alias("order_cart_rate_30d"),

            (
                pl.col("gmv_search_30d") /
                (pl.col("gmv_30d") + 1e-6)
            ).alias("search_gmv_share_30d"),

            (
                pl.col("gmv_cat_30d") /
                (pl.col("gmv_30d") + 1e-6)
            ).alias("cat_gmv_share_30d"),

            (
                pl.col("search_to_ord_30d") /
                (pl.col("to_ord_30d") + 1e-6)
            ).alias("search_order_share_30d"),

            (
                pl.col("cat_to_ord_30d") /
                (pl.col("to_ord_30d") + 1e-6)
            ).alias("cat_order_share_30d"),

            (
                (pl.col("gmv_7d") + 1.0) /
                (pl.col("gmv_30d") * 7.0 / 30.0 + 1.0)
            ).alias("gmv_7d_trend"),

            (
                (pl.col("to_ord_7d") + 1.0) /
                (pl.col("to_ord_30d") * 7.0 / 30.0 + 1.0)
            ).alias("orders_7d_trend"),

            (
                (pl.col("searches_7d") + 1.0) /
                (pl.col("searches_30d") * 7.0 / 30.0 + 1.0)
            ).alias("searches_7d_trend"),

            (
                (pl.col("to_cart_7d") + 1.0) /
                (pl.col("to_cart_30d") * 7.0 / 30.0 + 1.0)
            ).alias("cart_7d_trend"),

            (
                (pl.col("gmv_30d") + 1.0) /
                (pl.col("gmv_60d") - pl.col("gmv_30d") + 1.0)
            ).alias("gmv_30d_vs_previous_30d"),

            (
                (pl.col("to_ord_30d") + 1.0) /
                (pl.col("to_ord_60d") - pl.col("to_ord_30d") + 1.0)
            ).alias("orders_30d_vs_previous_30d"),

            (
                (pl.col("active_days_30d") + 1.0) /
                (pl.col("active_days_60d") - pl.col("active_days_30d") + 1.0)
            ).alias("activity_30d_vs_previous_30d"),

            (
                pl.col("lifetime_gmv") /
                (pl.col("lifetime_active_days") + 1e-6)
            ).alias("lifetime_gmv_per_active_day"),

            (
                pl.col("lifetime_to_ord") /
                (pl.col("lifetime_active_days") + 1e-6)
            ).alias("lifetime_orders_per_active_day"),

            (
                pl.col("lifetime_to_ord") /
                (pl.col("tenure_days") + 1.0)
            ).alias("orders_per_tenure_day"),

            (
                pl.col("lifetime_gmv") /
                (pl.col("tenure_days") + 1.0)
            ).alias("gmv_per_tenure_day"),

            (
                pl.col("to_ord_30d") /
                (pl.col("lifetime_to_ord") + 1.0)
            ).alias("orders_30d_lifetime_share"),

            (
                pl.col("to_ord_90d") /
                (pl.col("lifetime_to_ord") + 1.0)
            ).alias("orders_90d_lifetime_share"),

            (
                pl.col("gmv_30d") /
                (pl.col("lifetime_gmv") + 1.0)
            ).alias("gmv_30d_lifetime_share"),

            (
                pl.col("gmv_90d") /
                (pl.col("lifetime_gmv") + 1.0)
            ).alias("gmv_90d_lifetime_share"),
        )

        log_cols = [
            "gmv_7d",
            "gmv_30d",
            "gmv_60d",
            "gmv_90d",
            "to_ord_30d",
            "to_ord_90d",
            "to_cart_30d",
            "searches_30d",
            "lifetime_gmv",
            "lifetime_to_ord",
        ]

        features = features.with_columns(
            [
                pl.col(col)
                .clip(lower_bound=0)
                .log1p()
                .alias(f"log1p_{col}")
                for col in log_cols
            ]
        )

        features = features.with_columns(
            [
                pl.col(col).fill_null(0.0)
                for col in features.columns
                if col not in [
                    "user_id",
                    "first_activity_date",
                    "last_activity_date",
                    "last_order_date",
                    "last_cart_date",
                    "last_search_date",
                ]
            ]
        )

        features = features.with_columns(
            pl.col("days_since_last_activity").fill_null(9999.0),
            pl.col("days_since_last_order").fill_null(9999.0),
            pl.col("days_since_last_cart").fill_null(9999.0),
            pl.col("days_since_last_search").fill_null(9999.0),
            pl.col("tenure_days").fill_null(0.0),
            pl.lit(anchor).cast(pl.Date).alias("anchor_date"),
        )

        parts.append(features)

    return pl.concat(parts, how="diagonal_relaxed")

## Формирование целевой переменной

Для каждого пользователя считается сумма `gmv` в период от `anchor + 1` до `anchor + 30` дней.

Так моделируется ровно та величина, которую нужно предсказывать в соревновании.
Если в этом периоде пользователь не совершал покупок, его target равен нулю.

In [ ]:
def generate_targets(data, anchor_dates, user_ids, horizon=30):
    data_batch = data.filter(
        pl.col("user_id").is_in(user_ids)
    )

    parts = []

    for anchor in anchor_dates:
        target = (
            data_batch
            .filter(
                pl.col("event_date").is_between(
                    anchor + timedelta(days=1),
                    anchor + timedelta(days=horizon)
                )
            )
            .group_by("user_id")
            .agg(
                pl.col("gmv").sum().alias("target")
            )
            .with_columns(
                pl.lit(anchor).cast(pl.Date).alias("anchor_date")
            )
        )

        parts.append(target)

    result = pl.concat(parts, how="diagonal_relaxed")

    return result.with_columns(
        pl.col("target").fill_null(0.0)
    )

## Создание обучающих фолдов

Для каждой временной anchor-даты создаём признаки и соответствующий будущий target,
после чего объединяем их по `user_id` и `anchor_date`.

Результаты сохраняются отдельными Parquet-файлами: это позволяет повторно использовать
подготовленные данные и не пересчитывать признаки перед каждым экспериментом.

In [ ]:
for fold_idx, anchor in enumerate(anchors):
    fold_dir = FEATURES_DIR / f"fold_{fold_idx:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    for batch_idx in range(n_batches):
        batch_users = user_ids[
            batch_idx * BATCH_SIZE:
            (batch_idx + 1) * BATCH_SIZE
        ]

        features = generate_features(
            data,
            [anchor],
            batch_users
        )

        targets = generate_targets(
            data,
            [anchor],
            batch_users,
            HORIZON
        )

        fold = (
            features
            .join(
                targets,
                on=["anchor_date", "user_id"],
                how="left"
            )
            .with_columns(
                pl.col("target").fill_null(0.0)
            )
        )

        fold.write_parquet(
            fold_dir / f"batch_{batch_idx:04d}.parquet"
        )

    print(f"Fold {fold_idx}: {anchor}")

Fold 0: 2025-12-03
Fold 1: 2025-12-17
Fold 2: 2025-12-31
Fold 3: 2026-01-14


## Финальные признаки

Отдельно строим признаки на последнюю доступную дату — 13 февраля 2026 года.
Именно они будут переданы в финальную модель для прогноза периода
с 14 февраля по 15 марта 2026 года.

Target для этой таблицы не создаётся, поскольку будущие значения организаторами скрыты.

In [ ]:
final_dir = FEATURES_DIR / "fold_end"
final_dir.mkdir(parents=True, exist_ok=True)

for batch_idx in range(n_batches):
    batch_users = user_ids[
        batch_idx * BATCH_SIZE:
        (batch_idx + 1) * BATCH_SIZE
    ]

    features = generate_features(
        data,
        [anchor_end],
        batch_users
    )

    features.write_parquet(
        final_dir / f"batch_{batch_idx:04d}.parquet"
    )

print(f"Final features: {anchor_end}")

Final features: 2026-02-13


## Загрузка подготовленных данных

Загружаем временные фолды и финальную таблицу признаков.
Проверяем размеры таблиц, чтобы убедиться, что обработаны все пользователи
и что число признаков одинаково во всех наборах.

In [ ]:
def read_fold(fold_name):
    return pl.read_parquet(
        str(FEATURES_DIR / fold_name / "batch_*.parquet")
    )


folds = [
    read_fold(f"fold_{i:02d}")
    for i in range(N_FOLDS)
]

final_features = read_fold("fold_end")

print("Fold sizes:")
for i, fold in enumerate(folds):
    print(f"Fold {i}: {fold.shape}")

print(f"Final: {final_features.shape}")

Fold sizes:
Fold 0: (247929, 168)
Fold 1: (250000, 168)
Fold 2: (250000, 168)
Fold 3: (250000, 168)
Final: (250000, 167)


## Baseline-прогнозы

Сначала оцениваем простые стратегии без машинного обучения.

GMV за последние 30 дней используется как самый естественный baseline.
Для окон 7 и 14 дней значение масштабируется до 30-дневного горизонта,
чтобы сравнение было корректным.

In [ ]:
baseline_results = []

for fold_idx, fold in enumerate(folds):
    y = fold["target"].to_numpy()

    predictions = {
        "7d_scaled": fold["gmv_7d"].to_numpy() * 30.0 / 7.0,
        "14d_scaled": fold["gmv_14d"].to_numpy() * 30.0 / 14.0,
        "30d": fold["gmv_30d"].to_numpy(),
    }

    for name, pred in predictions.items():
        baseline_results.append({
            "fold": fold_idx,
            "model": name,
            "rmsle": rmsle(y, pred),
        })

baseline_results = pl.DataFrame(baseline_results)

print(baseline_results)
print(
    baseline_results
    .group_by("model")
    .agg(pl.mean("rmsle").alias("mean_rmsle"))
    .sort("mean_rmsle")
)

shape: (12, 3)
┌──────┬────────────┬──────────┐
│ fold ┆ model      ┆ rmsle    │
│ ---  ┆ ---        ┆ ---      │
│ i64  ┆ str        ┆ f64      │
╞══════╪════════════╪══════════╡
│ 0    ┆ 7d_scaled  ┆ 2.791484 │
│ 0    ┆ 14d_scaled ┆ 2.497379 │
│ 0    ┆ 30d        ┆ 2.226474 │
│ 1    ┆ 7d_scaled  ┆ 2.690654 │
│ 1    ┆ 14d_scaled ┆ 2.440211 │
│ …    ┆ …          ┆ …        │
│ 2    ┆ 14d_scaled ┆ 2.410811 │
│ 2    ┆ 30d        ┆ 2.22673  │
│ 3    ┆ 7d_scaled  ┆ 2.627089 │
│ 3    ┆ 14d_scaled ┆ 2.403393 │
│ 3    ┆ 30d        ┆ 2.195065 │
└──────┴────────────┴──────────┘
shape: (3, 2)
┌────────────┬────────────┐
│ model      ┆ mean_rmsle │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ 30d        ┆ 2.21622    │
│ 14d_scaled ┆ 2.437948   │
│ 7d_scaled  ┆ 2.693326   │
└────────────┴────────────┘


## Подготовка признаков для модели

Исключаем идентификаторы, даты и target, поскольку они не должны использоваться
как обычные числовые признаки.

Остальные столбцы содержат агрегаты поведения пользователя и передаются в CatBoost.

In [ ]:
DROP_COLS = [
    "user_id",
    "anchor_date",
    "target",
    "first_activity_date",
    "last_activity_date",
    "last_order_date",
    "last_cart_date",
    "last_search_date",
]

feature_cols = [
    col
    for col in folds[0].columns
    if col not in DROP_COLS
]

print(f"Features: {len(feature_cols)}")
print(feature_cols)

Features: 160
['search_7d', 'cat_7d', 'has_search_to_cart_7d', 'has_search_to_ord_7d', 'has_cat_to_cart_7d', 'has_cat_to_ord_7d', 'search_to_cart_7d', 'search_to_ord_7d', 'cat_to_cart_7d', 'cat_to_ord_7d', 'gmv_search_7d', 'gmv_cat_7d', 'to_cart_7d', 'to_ord_7d', 'gmv_7d', 'searches_7d', 'active_days_7d', 'max_gmv_7d', 'max_to_ord_7d', 'max_to_cart_7d', 'max_searches_7d', 'search_14d', 'cat_14d', 'has_search_to_cart_14d', 'has_search_to_ord_14d', 'has_cat_to_cart_14d', 'has_cat_to_ord_14d', 'search_to_cart_14d', 'search_to_ord_14d', 'cat_to_cart_14d', 'cat_to_ord_14d', 'gmv_search_14d', 'gmv_cat_14d', 'to_cart_14d', 'to_ord_14d', 'gmv_14d', 'searches_14d', 'active_days_14d', 'max_gmv_14d', 'max_to_ord_14d', 'max_to_cart_14d', 'max_searches_14d', 'search_30d', 'cat_30d', 'has_search_to_cart_30d', 'has_search_to_ord_30d', 'has_cat_to_cart_30d', 'has_cat_to_ord_30d', 'search_to_cart_30d', 'search_to_ord_30d', 'cat_to_cart_30d', 'cat_to_ord_30d', 'gmv_search_30d', 'gmv_cat_30d', 'to_cart_3

## Временная кросс-валидация CatBoost

Модель обучается на более ранних временных фолдах и проверяется на следующем фолде.
Такой порядок имитирует реальную задачу: предсказывать будущее, имея только прошлую историю.

Target преобразуется через `log1p`, потому что GMV содержит много нулей и имеет длинный
правый хвост. После прогнозирования выполняется обратное преобразование `expm1`.

In [ ]:
models = []
cv_results = []

for valid_idx in range(1, N_FOLDS):
    train_df = pl.concat(
        folds[:valid_idx],
        how="vertical_relaxed"
    )

    valid_df = folds[valid_idx]

    X_train = train_df.select(feature_cols).to_pandas()
    y_train = np.log1p(
        train_df["target"].to_numpy()
    )

    X_valid = valid_df.select(feature_cols).to_pandas()
    y_valid = np.log1p(
        valid_df["target"].to_numpy()
    )

    model = CatBoostRegressor(
        iterations=1500,
        depth=7,
        learning_rate=0.05,
        loss_function="RMSE",
        eval_metric="RMSE",
        l2_leaf_reg=5,
        random_seed=42,
        verbose=100,
        od_type="Iter",
        od_wait=150,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    pred_log = model.predict(X_valid)
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)

    score = rmsle(
        valid_df["target"].to_numpy(),
        pred
    )

    cv_results.append({
        "fold": valid_idx,
        "rmsle": score,
        "best_iteration": model.get_best_iteration() + 1,
    })

    models.append(model)

    print(
        f"Fold {valid_idx}: "
        f"RMSLE={score:.6f}, "
        f"best_iteration={model.get_best_iteration() + 1}"
    )

cv_results = pl.DataFrame(cv_results)

print(cv_results)
print(f"Mean RMSLE: {cv_results['rmsle'].mean():.6f}")
print(f"Median RMSLE: {cv_results['rmsle'].median():.6f}")

0:	learn: 2.3370918	test: 2.3058796	best: 2.3058796 (0)	total: 405ms	remaining: 10m 7s
100:	learn: 1.7527918	test: 1.7501394	best: 1.7501394 (100)	total: 33.1s	remaining: 7m 37s
200:	learn: 1.7454926	test: 1.7476556	best: 1.7476556 (200)	total: 1m 3s	remaining: 6m 47s
300:	learn: 1.7392522	test: 1.7468379	best: 1.7468315 (295)	total: 1m 32s	remaining: 6m 6s
400:	learn: 1.7338043	test: 1.7462436	best: 1.7462436 (400)	total: 1m 58s	remaining: 5m 24s
500:	learn: 1.7287708	test: 1.7457801	best: 1.7457801 (500)	total: 2m 24s	remaining: 4m 48s
600:	learn: 1.7239745	test: 1.7453549	best: 1.7453549 (600)	total: 2m 52s	remaining: 4m 17s
700:	learn: 1.7194733	test: 1.7450413	best: 1.7450413 (700)	total: 3m 19s	remaining: 3m 47s
800:	learn: 1.7150009	test: 1.7447435	best: 1.7447415 (799)	total: 3m 45s	remaining: 3m 16s
900:	learn: 1.7104194	test: 1.7445234	best: 1.7444616 (896)	total: 4m 13s	remaining: 2m 48s
1000:	learn: 1.7062032	test: 1.7443591	best: 1.7443122 (997)	total: 4m 42s	remaining: 2m

## Сравнение качества

Сравниваем средний RMSLE CatBoost с baseline-прогнозами.
Так можно проверить, даёт ли модель дополнительную пользу по сравнению
с использованием только недавнего GMV пользователя.

In [ ]:
catboost_mean = cv_results["rmsle"].mean()

baseline_mean = (
    baseline_results
    .group_by("model")
    .agg(pl.mean("rmsle").alias("rmsle"))
    .sort("rmsle")
)

print("Baseline:")
print(baseline_mean)

print(f"CatBoost mean RMSLE: {catboost_mean:.6f}")

Baseline:
shape: (3, 2)
┌────────────┬──────────┐
│ model      ┆ rmsle    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ 30d        ┆ 2.21622  │
│ 14d_scaled ┆ 2.437948 │
│ 7d_scaled  ┆ 2.693326 │
└────────────┴──────────┘
CatBoost mean RMSLE: 1.705864


## Важность признаков

Выводим признаки, которые сильнее всего влияют на предсказания модели.
Это помогает понять, какие характеристики поведения связаны с будущим GMV:
накопленные заказы, давность покупки, срок наблюдения и интенсивность активности.

In [ ]:
importance = models[-1].get_feature_importance()

feature_importance = (
    pl.DataFrame({
        "feature": feature_cols,
        "importance": importance,
    })
    .sort("importance", descending=True)
)

print(feature_importance.head(10))

shape: (10, 2)
┌─────────────────────────────┬────────────┐
│ feature                     ┆ importance │
│ ---                         ┆ ---        │
│ str                         ┆ f64        │
╞═════════════════════════════╪════════════╡
│ log1p_lifetime_to_ord       ┆ 8.392777   │
│ lifetime_to_ord             ┆ 8.348153   │
│ orders_per_tenure_day       ┆ 4.611982   │
│ gmv_per_tenure_day          ┆ 4.165191   │
│ tenure_days                 ┆ 3.73767    │
│ log1p_lifetime_gmv          ┆ 2.814572   │
│ days_since_last_order       ┆ 2.789987   │
│ to_ord_90d                  ┆ 2.497229   │
│ lifetime_search_to_ord      ┆ 2.238176   │
│ lifetime_gmv_per_active_day ┆ 2.22431    │
└─────────────────────────────┴────────────┘


## Выбор количества итераций

Берём медианное лучшее количество итераций по валидационным фолдам.
Медиана менее чувствительна к одному нестабильному фолду, чем выбор по единственному
временному периоду.

In [ ]:
final_iterations = int(
    np.median(
        cv_results["best_iteration"].to_numpy()
    )
)

print(f"Final iterations: {final_iterations}")

Final iterations: 1498


## Финальное обучение

После выбора параметров объединяем все доступные обучающие фолды
и обучаем одну финальную модель.

Используем количество итераций, выбранное на временной валидации,
чтобы сохранить найденный баланс между недообучением и переобучением.

In [ ]:
train_all = pl.concat(
    folds,
    how="vertical_relaxed"
)

X_train = train_all.select(feature_cols).to_pandas()
y_train = np.log1p(
    train_all["target"].to_numpy()
)

final_model = CatBoostRegressor(
    iterations=final_iterations,
    depth=7,
    learning_rate=0.05,
    loss_function="RMSE",
    l2_leaf_reg=5,
    random_seed=42,
    verbose=100,
)

final_model.fit(
    X_train,
    y_train,
)

0:	learn: 2.2844911	total: 1.12s	remaining: 27m 54s
100:	learn: 1.7181633	total: 1m 54s	remaining: 26m 19s
200:	learn: 1.7133594	total: 3m 30s	remaining: 22m 39s
300:	learn: 1.7105149	total: 5m 4s	remaining: 20m 12s
400:	learn: 1.7077605	total: 6m 43s	remaining: 18m 23s
500:	learn: 1.7054693	total: 8m 19s	remaining: 16m 33s
600:	learn: 1.7033605	total: 9m 53s	remaining: 14m 45s
700:	learn: 1.7013775	total: 11m 27s	remaining: 13m 1s
800:	learn: 1.6994686	total: 13m 1s	remaining: 11m 19s
900:	learn: 1.6976473	total: 14m 31s	remaining: 9m 37s
1000:	learn: 1.6958502	total: 16m 7s	remaining: 8m
1100:	learn: 1.6941910	total: 17m 38s	remaining: 6m 21s
1200:	learn: 1.6925614	total: 19m 13s	remaining: 4m 45s
1300:	learn: 1.6909025	total: 20m 48s	remaining: 3m 9s
1400:	learn: 1.6892896	total: 22m 24s	remaining: 1m 33s
1497:	learn: 1.6877605	total: 23m 51s	remaining: 0us


CatBoostRegressor(depth=7, iterations=1498, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

## Финальный прогноз

Применяем модель к признакам пользователей на 13 февраля 2026 года.
Предсказания переводятся из логарифмической шкалы обратно в исходную шкалу GMV
с помощью `expm1`.

Отрицательные значения заменяются нулём, поскольку GMV не может быть отрицательным.

In [ ]:
X_test = final_features.select(feature_cols).to_pandas()

pred_log = final_model.predict(X_test)

pred = np.expm1(pred_log)
pred = np.clip(pred, 0, None)

print(f"Predictions: {len(pred):,}")
print(f"Min: {pred.min():.6f}")
print(f"Median: {np.median(pred):.6f}")
print(f"Mean: {pred.mean():.6f}")
print(f"Max: {pred.max():.6f}")

Predictions: 250,000
Min: 0.000000
Median: 6.397524
Mean: 35.428794
Max: 3543.198657


## Формирование файла отправки

Создаём таблицу из идентификатора пользователя и его прогноза.
Для каждого из 250 000 пользователей должна быть ровно одна строка.

In [ ]:
submission = (
    final_features
    .select("user_id")
    .with_columns(
        pl.Series("predict", pred)
    )
    .select(["user_id", "predict"])
)

print(f"Rows: {submission.height:,}")
print(f"Unique users: {submission['user_id'].n_unique():,}")
print(f"Null predictions: {submission['predict'].null_count()}")
print(
    f"Negative predictions: "
    f"{submission.filter(pl.col('predict') < 0).height}"
)

submission.head()

Rows: 250,000
Unique users: 250,000
Null predictions: 0
Negative predictions: 0


user_id,predict
i64,f64
91523,82.32597
171880,21.628065
26106,0.377403
158396,10.931395
43937,31.977602


## Проверка и сохранение результата

Перед сохранением проверяем количество строк, уникальность пользователей,
отсутствие пропусков и отрицательных предсказаний.

После проверки сохраняем итоговый CSV-файл, который можно использовать
для отправки в соревновании.

In [ ]:
SUBMISSION_PATH = BASE_DIR / "data/submission_catboost_v1.csv"

submission.write_csv(SUBMISSION_PATH)

print(f"Saved: {SUBMISSION_PATH}")

Saved: /content/drive/MyDrive/Colab Notebooks/e-cup/data/submission_catboost_v1.csv
